# LiteFNO Training (Google Colab)

This notebook sets up the `litefno-repro` repository, downloads a sample dataset, preprocesses it, and runs a short training job.

**Tip:** Use a GPU runtime in Colab for faster training.


## 1. Clone the repository

If you opened this notebook directly from GitHub in Colab, the repo is already present. Otherwise, this cell will clone it into `/content`.


In [ ]:
from pathlib import Path

repo_dir = Path('/content/litefno-repro')
if not repo_dir.exists():
    !git clone https://github.com/jgx0/litefno-repro.git {repo_dir}
%cd /content/litefno-repro


## 2. Install dependencies

Colab already includes PyTorch, but you can install a specific CUDA build if needed.


In [ ]:
# Optional: install a CUDA-specific PyTorch build
# !pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121

!pip install -q -e .
!pip install -q the-well-download ipywidgets


## 3. Configure the run

Use the widgets below to select dataset/experiment configs, training epochs, device, and whether to download or preprocess data.


In [ ]:
import ipywidgets as widgets
from pathlib import Path
from IPython.display import display

try:
    from google.colab import output
    output.enable_custom_widget_manager()
except Exception:
    pass

dataset_options = sorted(Path('configs/datasets').glob('*.yaml'))
experiment_options = sorted(Path('configs/experiments').glob('*.yaml'))
dataset_options = [str(path) for path in dataset_options]
experiment_options = [str(path) for path in experiment_options]

default_dataset = 'configs/datasets/gray_scott_reaction_diffusion.yaml'
if default_dataset not in dataset_options and dataset_options:
    default_dataset = dataset_options[0]

default_experiment = 'configs/experiments/litefno_gray_scott_reaction_diffusion.yaml'
if default_experiment not in experiment_options and experiment_options:
    default_experiment = experiment_options[0]

dataset_dropdown = widgets.Dropdown(
    options=dataset_options,
    value=default_dataset if dataset_options else None,
    description='Dataset',
    layout=widgets.Layout(width='80%'),
)
experiment_dropdown = widgets.Dropdown(
    options=experiment_options,
    value=default_experiment if experiment_options else None,
    description='Experiment',
    layout=widgets.Layout(width='80%'),
)
epochs_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=200,
    step=1,
    description='Epochs',
    continuous_update=False,
)
device_dropdown = widgets.Dropdown(
    options=[('Auto', 'auto'), ('CPU', 'cpu'), ('CUDA', 'cuda')],
    value='auto',
    description='Device',
)
download_checkbox = widgets.Checkbox(value=True, description='Download data')
preprocess_checkbox = widgets.Checkbox(value=True, description='Preprocess data')
resume_text = widgets.Text(value='', description='Resume from', placeholder='/path/to/epoch_0100.pt')

display(widgets.VBox([
    dataset_dropdown,
    experiment_dropdown,
    epochs_slider,
    device_dropdown,
    download_checkbox,
    preprocess_checkbox,
    resume_text,
]))


## 4. Download and preprocess data

Run this cell after updating the widget values above.


In [ ]:
import shlex
import subprocess

def run(cmd_parts):
    cmd = ' '.join(shlex.quote(part) for part in cmd_parts)
    print(f'$ {cmd}')
    subprocess.run(cmd, shell=True, check=True)

if dataset_dropdown.value:
    if download_checkbox.value:
        run(['litefno', 'download', '--config', dataset_dropdown.value])
    if preprocess_checkbox.value:
        run(['litefno', 'preprocess', '--config', dataset_dropdown.value])
else:
    raise ValueError('No dataset config found in configs/datasets.')


## 5. Train LiteFNO

Run a short training loop to validate setup. Increase `Epochs` for a full run.


In [ ]:
import torch

device_choice = device_dropdown.value
if device_choice == 'auto':
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
else:
    device = device_choice

train_cmd = [
    'litefno',
    'train',
    '--config',
    experiment_dropdown.value,
    '--set',
    f'training.epochs={int(epochs_slider.value)}',
    '--set',
    f'training.device={device}',
]

resume_from = resume_text.value.strip()
if resume_from:
    train_cmd += ['--set', f'training.resume_from={resume_from}']

run(train_cmd)
